In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_TELEMETRY", "true")   # nada sale a internet salvo Databricks si se configura
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "src"))   # los notebooks importan de src/, nunca al revés

import numpy as np
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)

# 04 · Variables — cada idea es una hipótesis medida

**Cómo se mide:** LightGBM con parámetros fijos (los de partida del torneo), los mismos 5 pliegues por cliente, semilla 42.

- **Métrica de decisión: PR-AUC** de la clase "en mora". La métrica de negocio (recall con precisión ≥ 20 %) es muy
  ruidosa cuando la señal es débil (salta de 0 a 0,3 entre pliegues porque la precisión exigida queda cerca del
  máximo alcanzable); PR-AUC mide lo mismo —ordenar bien la mora— sin depender de un único umbral.
- **Control:** la métrica de negocio y el AUC.
- **Regla escrita antes de medir:** una idea se conserva si mejora la PR-AUC en **al menos 1 vez el ruido** entre
  pliegues (desviación de la diferencia pareada) **y** no empeora la métrica de negocio.

In [2]:
import json
import joblib
from proyecto.config import GROUP_COL, TARGET, MIN_PRECISION, REPORTS_DIR
from proyecto.data.split import group_cv
from proyecto.models.train import build_pipeline, candidates, evaluate_folds, split_xy, summarize
from proyecto.tracking.mlflow_utils import configure_mlflow, log_run, set_experiment

train = joblib.load(ROOT / "data" / "processed" / "train.joblib")
X, y = split_xy(train)
cv = group_cv(5)   # los MISMOS 5 pliegues por cliente para todo (semilla 42)
donde = configure_mlflow()
print(f"entrenamiento: {len(train):,} créditos · mora {y.mean():.1%} · MLflow: {donde}")

entrenamiento: 36,857 créditos · mora 16.1% · MLflow: local


In [3]:
from proyecto.features.engineering import IDEAS
set_experiment("libranzaradar-04-features")
modelo, escala = candidates()["lightgbm"]
variantes = {"0 · referencia (condiciones del crédito)": ()}
variantes.update({f"{i + 1} · + {nombre}": (nombre,) for i, nombre in enumerate(IDEAS)})
variantes["5 · todas las ideas"] = tuple(IDEAS)
por_pliegue = {}
for nombre, ideas in variantes.items():
    f = evaluate_folds(build_pipeline(modelo, escala, ideas=ideas), X, y, cv)
    por_pliegue[nombre] = f
    s = summarize(f)
    log_run(nombre, {"ideas": ",".join(ideas) or "ninguna", "modelo": "lightgbm"}, s, tags={"etapa": "04_features"})
    print(f"{nombre:45s} PR-AUC {s['pr_auc_mean']:.4f} ± {s['pr_auc_std']:.4f} · negocio {s['recall_at_p_mean']:.3f} ± {s['recall_at_p_std']:.3f}")

2026/10/30 21:23:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/30 21:23:31 INFO mlflow.store.db.utils: Updating database tables


2026/10/30 21:23:32 INFO mlflow.tracking.fluent: Experiment with name 'libranzaradar-04-features' does not exist. Creating a new experiment.


0 · referencia (condiciones del crédito)      PR-AUC 0.1800 ± 0.0041 · negocio 0.186 ± 0.075


1 · + razones                                 PR-AUC 0.1808 ± 0.0053 · negocio 0.143 ± 0.088


2 · + historial                               PR-AUC 0.1795 ± 0.0052 · negocio 0.167 ± 0.108


3 · + empresa                                 PR-AUC 0.1780 ± 0.0036 · negocio 0.123 ± 0.036


4 · + edad                                    PR-AUC 0.1773 ± 0.0042 · negocio 0.098 ± 0.083


5 · todas las ideas                           PR-AUC 0.1781 ± 0.0030 · negocio 0.107 ± 0.062


In [4]:
ref = por_pliegue["0 · referencia (condiciones del crédito)"]
filas = []
for nombre, f in por_pliegue.items():
    d = f["pr_auc"] - ref["pr_auc"]
    ruido = d.std(ddof=1)
    mejora_ruido = d.mean() / ruido if ruido > 0 else 0.0
    d_neg = (f["recall_at_p"] - ref["recall_at_p"]).mean()
    conservar = (mejora_ruido >= 1) and (d_neg >= 0) and not nombre.startswith("0")
    filas.append({"variante": nombre, "PR-AUC": f"{f['pr_auc'].mean():.4f} ± {f['pr_auc'].std(ddof=1):.4f}",
                  "AUC": round(f["roc_auc"].mean(), 4), "negocio (recall@P≥20%)": f"{f['recall_at_p'].mean():.3f} ± {f['recall_at_p'].std(ddof=1):.3f}",
                  "mejora PR-AUC": round(d.mean(), 4), "mejora / ruido": round(mejora_ruido, 2),
                  "decisión": "referencia" if nombre.startswith("0") else ("conservar" if conservar else "descartar")})
tabla = pd.DataFrame(filas)
tabla.to_csv(REPORTS_DIR / "04_features_tabla.csv", index=False)
tabla

,variante,PR-AUC,AUC,negocio (recall@P≥20%),mejora PR-AUC,mejora / ruido,decisión
0,0 · referencia (condiciones del crédito),0.1800 ± 0.0041,0.5326,0.186 ± 0.075,0.0000,0.00,referencia
1,1 · + razones,0.1808 ± 0.0053,0.5369,0.143 ± 0.088,0.0008,0.47,descartar
2,2 · + historial,0.1795 ± 0.0052,0.5310,0.167 ± 0.108,-0.0005,-0.43,descartar
3,3 · + empresa,0.1780 ± 0.0036,0.5310,0.123 ± 0.036,-0.0020,-1.16,descartar
4,4 · + edad,0.1773 ± 0.0042,0.5288,0.098 ± 0.083,-0.0027,-2.45,descartar
5,5 · todas las ideas,0.1781 ± 0.0030,0.5312,0.107 ± 0.062,-0.0019,-0.93,descartar


**Lectura:** ninguna idea supera su propio ruido de forma consistente. Las variables financieras derivadas, el
historial del cliente, la empresa pagadora (codificada con su tasa de mora de entrenamiento) y la edad no mejoran
de forma distinguible a la referencia. **El pipeline final usa solo la referencia**: 12 variables conocidas al aprobar,
lo que además simplifica la app (menos campos que pedir) y evita usar la edad, un atributo de uso regulado.

In [5]:
from proyecto.features.engineering import feature_set, raw_input_columns
conservadas = [ideas for ideas, decision in zip(variantes.values(), tabla["decisión"]) if decision == "conservar"]
FINAL_IDEAS = tuple(dict.fromkeys(i for ideas in conservadas for i in ideas))
final = {"ideas": list(FINAL_IDEAS), "columnas": raw_input_columns(feature_set(FINAL_IDEAS))}
(REPORTS_DIR / "04_features_finales.json").write_text(json.dumps(final, indent=2, ensure_ascii=False), encoding="utf-8")
final

{'ideas': [],
 'columnas': ['term_months',
  'loan_amount',
  'monthly_installment',
  'interest_rate_ea',
  'due_day',
  'has_guarantee',
  'guarantee_value',
  'has_cosigner',
  'product',
  'business_type',
  'economic_destination',
  'city']}